# Notebook 05: Online vs Batch Inference

## Why This Matters

The choice between online (real-time) and batch inference is one of the first architectural decisions in any ML system design interview, and it ripples through every subsequent choice—feature freshness, model complexity, infrastructure cost, and latency SLAs. LinkedIn uses batch inference for some recommendation pre-computations, while serving real-time models for feed ranking. Stripe's fraud detection must be online (real-time) by definition. DoorDash uses a hybrid approach where estimated delivery times use online inference but restaurant recommendations may be pre-computed. Getting this distinction right—and knowing the hybrid patterns—is a strong signal of ML systems experience.

## 1. Online vs Batch Inference: The Fundamental Tradeoff

Online inference computes predictions at request time for individual inputs. Batch inference computes predictions for large sets of inputs ahead of time and stores the results for lookup. The key tradeoff is latency, freshness, and infrastructure cost.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch

fig, axes = plt.subplots(1, 2, figsize=(16, 9))

def box(ax, x, y, w, h, text, color, fontsize=9):
    r = FancyBboxPatch((x,y), w, h, boxstyle="round,pad=0.12",
                       facecolor=color, edgecolor='white', linewidth=1.8, alpha=0.9)
    ax.add_patch(r)
    ax.text(x+w/2, y+h/2, text, ha='center', va='center',
            fontsize=fontsize, fontweight='bold', color='white', multialignment='center')

def arr(ax, x1, y1, x2, y2, label='', color='#666'):
    ax.annotate('', xy=(x2,y2), xytext=(x1,y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=2))
    if label:
        ax.text((x1+x2)/2, (y1+y2)/2+0.1, label, fontsize=8, ha='center', color=color)

# Online inference
ax = axes[0]
ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.axis('off')
ax.set_title('Online (Real-Time) Inference', fontsize=12, fontweight='bold', color='#2980B9')

box(ax, 3, 8.5, 4, 1, 'User Request', '#7F8C8D')
box(ax, 1, 6.5, 3.5, 1.2, 'Feature Retrieval\n(online store, <10ms)', '#E67E22')
box(ax, 5.5, 6.5, 3.5, 1.2, 'Model Inference\n(GPU/CPU, <50ms)', '#2980B9')
box(ax, 3, 4.5, 4, 1, 'Response (<100ms)', '#27AE60')

arr(ax, 5, 8.5, 2.75, 7.7)
arr(ax, 5, 8.5, 7.25, 7.7)
arr(ax, 2.75, 6.5, 3, 5.5, '')
arr(ax, 7.25, 6.5, 7, 5.5, '')

pros = [
    ('✓', 'Uses most recent features', '#27AE60'),
    ('✓', 'Can use request-time context', '#27AE60'),
    ('✓', 'Handles any user/item', '#27AE60'),
    ('✗', 'High infrastructure cost', '#E74C3C'),
    ('✗', 'Latency budget constraint', '#E74C3C'),
    ('✗', 'Model must be fast', '#E74C3C'),
]
ax.text(5, 3.8, 'Characteristics:', fontsize=9, fontweight='bold', color='#333', ha='center')
for i, (icon, text, color) in enumerate(pros):
    ax.text(0.5, 3.2 - i * 0.42, f'{icon} {text}', fontsize=8.5, color=color)

ax.text(5, 0.8, 'Examples: Fraud detection, feed ranking,\nad CTR prediction, ETA calculation',
        ha='center', fontsize=8.5, style='italic', color='#555',
        bbox=dict(boxstyle='round', facecolor='#EBF5FB', alpha=0.8))

# Batch inference
ax2 = axes[1]
ax2.set_xlim(0, 10); ax2.set_ylim(0, 10); ax2.axis('off')
ax2.set_title('Batch (Pre-computed) Inference', fontsize=12, fontweight='bold', color='#8E44AD')

box(ax2, 1, 8.5, 3.5, 1, 'Training Data\n(all users/items)', '#7F8C8D')
box(ax2, 5.5, 8.5, 3.5, 1, 'Scheduled Pipeline\n(hourly/daily)', '#E74C3C')
box(ax2, 3, 6.5, 4, 1.2, 'Batch Model Inference\n(Spark / Dataflow)', '#8E44AD')
box(ax2, 3, 4.8, 4, 1, 'Prediction Store\n(Redis / DynamoDB)', '#2C3E50')
box(ax2, 3, 3, 4, 1.2, 'User Request →\nLookup by key (<5ms)', '#27AE60')

arr(ax2, 2.75, 8.5, 4, 7.7)
arr(ax2, 7.25, 8.5, 6, 7.7)
arr(ax2, 5, 6.5, 5, 5.8)
arr(ax2, 5, 4.8, 5, 4.2)

pros2 = [
    ('✓', 'Very low serving latency (cache hit)', '#27AE60'),
    ('✓', 'Cheap at serving time', '#27AE60'),
    ('✓', 'Can use expensive models', '#27AE60'),
    ('✗', 'Stale predictions (up to hours old)', '#E74C3C'),
    ('✗', 'Only known users/items', '#E74C3C'),
    ('✗', 'Cold start problem', '#E74C3C'),
]
ax2.text(5, 2.3, 'Characteristics:', fontsize=9, fontweight='bold', color='#333', ha='center')
for i, (icon, text, color) in enumerate(pros2):
    ax2.text(0.5, 1.9 - i * 0.37, f'{icon} {text}', fontsize=8.5, color=color)

ax2.text(5, 0.2, 'Examples: Weekly playlist generation, email\nrecommendations, offline scoring campaigns',
         ha='center', fontsize=8.5, style='italic', color='#555',
         bbox=dict(boxstyle='round', facecolor='#F5EEF8', alpha=0.8))

plt.tight_layout()
plt.savefig('online_vs_batch.png', dpi=120, bbox_inches='tight')
plt.show()

## 2. Decision Framework: Choosing Between Online and Batch

The decision is not always obvious. Use this framework to reason through the tradeoffs systematically.

In [ ]:
decision_factors = {
    "Use ONLINE inference when...": [
        ("Latency budget < 500ms", "Batch lookup is fast but pipeline lag may exceed SLA"),
        ("Context is request-specific", "User query, current timestamp, session state are inputs"),
        ("Feature freshness is critical", "Fraud detection: last-1hr transaction features matter"),
        ("Catalog changes frequently", "New products added hourly can't wait for batch rerun"),
        ("Cold start users are common", "New users need immediate recommendations"),
        ("QPS is moderate", "High QPS × expensive model = costs explode"),
    ],
    "Use BATCH inference when...": [
        ("Latency budget > 1 second", "Email campaigns, push notifications, weekly digests"),
        ("Model is expensive to run", "Large two-tower retrieval, XGBoost on 1000 features"),
        ("Predictions are user-stable", "Weekly playlist: user taste doesn't change in 5 minutes"),
        ("Entity set is bounded", "All known users/items can be scored ahead of time"),
        ("Cost is the primary constraint", "Startup or cost-sensitive product"),
        ("Offline analysis is primary use case", "Business reports, audience segmentation"),
    ],
    "Use HYBRID inference when...": [
        ("Use batch for candidates, online for ranking", "LinkedIn: pre-compute candidate connections, rank in real-time"),
        ("Pre-compute embeddings, online for similarity", "Store item embeddings offline; compute dot-product online"),
        ("Batch for known users, online for new users", "Fallback to online when batch prediction is missing"),
        ("Pre-compute coarse scores, refine online", "Retrieval from batch cache → fine-grained online re-ranking"),
    ],
}

for category, items in decision_factors.items():
    print(f"\n{'='*65}")
    print(f"  {category}")
    print(f"{'='*65}")
    for criterion, rationale in items:
        print(f"  • {criterion}")
        print(f"      → {rationale}")

## 3. Hybrid Inference Architecture: The Two-Stage Pattern

The dominant pattern at scale combines batch pre-computation with online refinement. The batch stage handles expensive retrieval over large catalogs; the online stage handles personalization with fresh context. This is used by YouTube, LinkedIn, Pinterest, and most large recommendation systems.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(16, 10))
ax.set_xlim(0, 16); ax.set_ylim(0, 10); ax.axis('off')
ax.set_title('Hybrid Inference: Two-Stage Architecture', fontsize=14, fontweight='bold')

def box(ax, x, y, w, h, text, color, fontsize=8.5):
    r = FancyBboxPatch((x,y), w, h, boxstyle="round,pad=0.12",
                       facecolor=color, edgecolor='white', linewidth=1.8, alpha=0.9)
    ax.add_patch(r)
    ax.text(x+w/2, y+h/2, text, ha='center', va='center',
            fontsize=fontsize, fontweight='bold', color='white', multialignment='center')

def arr(ax, x1, y1, x2, y2, label='', color='#666'):
    ax.annotate('', xy=(x2,y2), xytext=(x1,y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.8))
    if label:
        mx, my = (x1+x2)/2, (y1+y2)/2
        ax.text(mx, my+0.15, label, fontsize=7.5, ha='center', color=color)

# OFFLINE PATH (top)
ax.text(8, 9.5, '── OFFLINE PATH (runs every few hours) ──', ha='center', fontsize=10,
        color='#E67E22', fontweight='bold')

box(ax, 0.3, 7.8, 2.5, 1.2, 'Item Catalog\n(1B+ items)', '#7F8C8D')
box(ax, 3.5, 7.8, 3, 1.2, 'Item Embedding\nModel (batch)', '#E67E22')
box(ax, 7.5, 7.8, 3, 1.2, 'ANN Index\n(Faiss/ScaNN)', '#E74C3C')
box(ax, 11.5, 7.8, 3, 1.2, 'User Candidate\nCache (Redis)', '#8E44AD')

arr(ax, 2.8, 8.4, 3.5, 8.4)
arr(ax, 6.5, 8.4, 7.5, 8.4)
arr(ax, 10.5, 8.4, 11.5, 8.4)

box(ax, 0.3, 6.0, 2.5, 1.2, 'User History\n(batch features)', '#2980B9')
box(ax, 3.5, 6.0, 3, 1.2, 'User Embedding\nModel (batch)', '#3498DB')
arr(ax, 2.8, 6.6, 3.5, 6.6)
arr(ax, 6.5, 6.6, 8, 7.8)

# ONLINE PATH (bottom)
ax.text(8, 5.2, '── ONLINE PATH (<100ms budget) ──', ha='center', fontsize=10,
        color='#2980B9', fontweight='bold')

box(ax, 0.3, 3.2, 2.2, 1.2, 'User\nRequest', '#7F8C8D')
box(ax, 3.2, 3.2, 2.8, 1.2, 'Candidate\nRetrieval\n(cache lookup)', '#8E44AD')
box(ax, 6.8, 3.2, 2.8, 1.2, 'Feature\nAssembly\n(online store)', '#E67E22')
box(ax, 10.2, 3.2, 2.8, 1.5, 'Ranking\nModel\n(online, <30ms)', '#2980B9')
box(ax, 13.5, 3.2, 2.2, 1.2, 'Response\n(top-K items)', '#27AE60')

arr(ax, 2.5, 3.8, 3.2, 3.8)
arr(ax, 6.0, 3.8, 6.8, 3.8)
arr(ax, 9.6, 3.8, 10.2, 3.8)
arr(ax, 13.0, 3.8, 13.5, 3.8)

# Cache feeds online path
arr(ax, 13, 7.8, 4.6, 4.4, '~1000 candidates', '#8E44AD')

# Budget annotations
budgets = [
    (4.6, 2.6, '~5ms'),
    (8.2, 2.6, '~15ms'),
    (11.6, 2.6, '~30ms'),
    (14.6, 2.6, '~5ms'),
]
for x, y, label in budgets:
    ax.text(x, y, label, ha='center', fontsize=8, color='#666', style='italic')

ax.text(8, 2.0, 'Total online budget: ~55ms | Leaves 45ms for network + other overhead within 100ms SLA',
        ha='center', fontsize=9, color='#27AE60', style='italic')

# Numbers
ax.text(4.6, 4.8, '1B items →', ha='center', fontsize=8, color='#E74C3C')
ax.text(8.2, 4.8, '→ ~1000 candidates →', ha='center', fontsize=8, color='#8E44AD')
ax.text(11.6, 4.8, '→ top 50 →', ha='center', fontsize=8, color='#2980B9')
ax.text(14, 4.8, '→ top 10', ha='center', fontsize=8, color='#27AE60')

plt.tight_layout()
plt.savefig('hybrid_inference.png', dpi=120, bbox_inches='tight')
plt.show()

print("Key numbers for the two-stage recommendation system:")
print("  Catalog size:     1 billion items")
print("  After retrieval:  ~1,000 candidates (ANN index lookup)")
print("  After ranking:    ~50 candidates")
print("  Shown to user:    10-20 items")
print("\nThe batch ANN index makes it feasible to search 1B items in ~5ms.")
print("Exhaustive online search of 1B items is impossible in 100ms budget.")

## 4. Measuring Freshness: The Staleness Problem

Batch predictions become stale over time. Quantifying the impact of staleness—and setting the right recomputation frequency—requires understanding your domain's data volatility.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

time_hours = np.linspace(0, 48, 1000)

# Different decay functions for different domains
def exponential_decay(t, half_life_hours):
    return np.exp(-np.log(2) * t / half_life_hours)

domains = [
    ('Fraud detection signals', 0.5, '#E74C3C', 'Needs real-time only'),
    ('News feed trending', 2, '#E67E22', 'Batch if < 15min'),
    ('Social connections', 12, '#F39C12', 'Hourly batch OK'),
    ('Movie recommendations', 48, '#27AE60', 'Daily batch fine'),
    ('User demographics', 720, '#2980B9', 'Weekly batch fine'),
]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

ax = axes[0]
for name, half_life, color, recommendation in domains:
    relevance = exponential_decay(time_hours, half_life)
    ax.plot(time_hours, relevance, linewidth=2, color=color,
            label=f'{name} (t½={half_life}h)')

ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.7, label='50% relevance threshold')
ax.axvline(x=1, color='gray', linestyle=':', alpha=0.5, label='1 hour mark')
ax.axvline(x=24, color='gray', linestyle='-.', alpha=0.5, label='24 hour mark')
ax.set_xlabel('Hours since prediction computed', fontsize=11)
ax.set_ylabel('Prediction Relevance (1.0 = fresh)', fontsize=11)
ax.set_title('Prediction Staleness by Domain\n(Exponential decay model)', fontsize=11, fontweight='bold')
ax.legend(fontsize=8, loc='upper right')
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.05)

# Recomputation frequency recommendations
ax2 = axes[1]
ax2.set_xlim(0, 10); ax2.set_ylim(0, len(domains) + 0.5); ax2.axis('off')
ax2.set_title('Recommended Inference Strategy by Domain', fontsize=11, fontweight='bold')

from matplotlib.patches import FancyBboxPatch
strategy_colors = {'Online only': '#E74C3C', 'Batch (every 15 min)': '#E67E22',
                   'Batch (hourly)': '#F39C12', 'Batch (daily)': '#27AE60', 'Batch (weekly)': '#2980B9'}

strategies = [
    ('Fraud detection signals', 'Online only', '< 30s half-life'),
    ('News feed trending', 'Batch (every 15 min)', '2h half-life'),
    ('Social connections', 'Batch (hourly)', '12h half-life'),
    ('Movie recommendations', 'Batch (daily)', '48h half-life'),
    ('User demographics', 'Batch (weekly)', '720h half-life'),
]

for i, (domain, strategy, reason) in enumerate(strategies):
    y = len(strategies) - i - 0.3
    color = [v for k, v in strategy_colors.items() if k == strategy][0]
    r = FancyBboxPatch((0.1, y), 9.8, 0.7, boxstyle="round,pad=0.08",
                       facecolor=color, alpha=0.15, edgecolor=color, linewidth=1.5)
    ax2.add_patch(r)
    ax2.text(0.5, y+0.35, domain, fontsize=9, va='center', color='#333', fontweight='bold')
    ax2.text(5.5, y+0.5, strategy, fontsize=9, va='center', color=color, fontweight='bold')
    ax2.text(5.5, y+0.2, reason, fontsize=8, va='center', color='#666')

plt.tight_layout()
plt.savefig('staleness_analysis.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Near-Real-Time Inference: The Middle Ground

Near-real-time inference (sometimes called 'micro-batch') fills the gap between full online (per-request) and batch (hourly/daily). It processes windows of data continuously—every 1-15 minutes—using streaming frameworks like Flink or Spark Structured Streaming.

In [ ]:
import pandas as pd

comparison = pd.DataFrame([
    {
        'Inference Type': 'Online (per-request)',
        'Latency': '< 100ms',
        'Freshness': 'Real-time',
        'Throughput': 'Bounded by model speed',
        'Infrastructure': 'Model servers + feature store',
        'Cost at Scale': 'Highest',
        'Complexity': 'High',
        'Use Cases': 'Fraud, ads, search, real-time ranking',
    },
    {
        'Inference Type': 'Near-real-time (micro-batch)',
        'Latency': '1-15 minutes',
        'Freshness': 'Minutes old',
        'Throughput': 'Very high (batch GPU)',
        'Infrastructure': 'Flink + batch scorer',
        'Cost at Scale': 'Medium',
        'Complexity': 'Medium',
        'Use Cases': 'Feed refresh, trending content, push notifications',
    },
    {
        'Inference Type': 'Batch (scheduled)',
        'Latency': 'Hours',
        'Freshness': 'Hours to days old',
        'Throughput': 'Highest (Spark)',
        'Infrastructure': 'Spark cluster + prediction store',
        'Cost at Scale': 'Lowest per prediction',
        'Complexity': 'Low',
        'Use Cases': 'Email campaigns, weekly playlists, offline analysis',
    },
])

print("Three-tier inference comparison:")
for col in comparison.columns:
    print(f"\n{'='*60}")
    print(f"  {col}")
    for _, row in comparison.iterrows():
        print(f"  [{row['Inference Type'][:20]:22s}] {row[col]}")

print("\n" + "="*60)
print("Decision heuristic:")
print("  If latency SLA < 500ms:    → Online inference")
print("  If staleness < 30 minutes: → Near-real-time (micro-batch)")
print("  If staleness < 24 hours:   → Batch (hourly/daily)")
print("  If staleness OK daily:     → Batch (daily/weekly)")
print("\nMost large systems use ALL THREE for different features/use cases.")

## Summary

| Dimension | Online | Near-Real-Time | Batch |
|-----------|--------|----------------|-------|
| Latency | <100ms | 1-15 min | Hours |
| Freshness | Real-time | Minutes stale | Hours stale |
| Infrastructure | Model server + online features | Streaming + batch scorer | Spark + prediction store |
| Cost per prediction | Highest | Medium | Lowest |
| Cold start handling | Yes | Partial | No |
| Model complexity | Constrained | Moderate | Unconstrained |

**Key Patterns**
- Two-stage retrieval is the dominant hybrid pattern: batch ANN index → online ranker
- Most production systems use all three tiers for different signals
- Staleness matters by domain: fraud detection needs real-time; weekly playlists can be batch
- Cold start = always use online (or a fallback model) since batch predictions won't exist
- Near-real-time (micro-batch streaming) is underutilized and often the best cost-freshness tradeoff